# ITRI626 Practical Project: 120-Breed Dog Classification
## Model: EfficientNetV2 (Compound-Scaled CNN with Transfer Learning)

**Student**: JP  
**Module**: ITRI626 Deep Learning for Image Classification  
**Dataset**: Stanford Dogs Dataset (120 Breeds, ~20,580 images)  
**Hardware**: NVIDIA GeForce RTX 4050 Laptop GPU (6GB VRAM)  

---

### 1. Environment & Hardware Setup
We verify the computing environment, set the random seed to `42` for complete reproducibility (Rubric Section 7.1), and check GPU availability.

In [ ]:
import os
import sys
import json
import random
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision

# Ensure project imports work cleanly
cwd = Path.cwd()
if str(cwd) not in sys.path:
    sys.path.append(str(cwd))

from dataset import get_dataloaders, clean_breed_name
from model import get_model
from train import set_seed, plot_training_curves
from evaluate import evaluate_checkpoint

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch Version: {torch.__version__}')
print(f'Compute Device: {device}')
if device.type == 'cuda':
    print(f'GPU Hardware: {torch.cuda.get_device_name(0)}')
    print(f'Available VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB')

### 2. Dataset Exploration & Class Distributions (Rubric Section 5)
We load the 120-breed dataset from `archive/images/Images`, verify class distributions, and inspect sample dog images.

In [ ]:
workspace_root = cwd.parent if cwd.name == 'JP_Model' else cwd
train_loader, val_loader, test_loader, class_names = get_dataloaders(
    workspace_root=workspace_root,
    batch_size=32 if torch.cuda.is_available() else 16,
    seed=42
)

n_train = len(train_loader.dataset)
n_val = len(val_loader.dataset)
n_test = len(test_loader.dataset)
n_total = n_train + n_val + n_test

print(f'Total Classes (Dog Breeds): {len(class_names)}')
print(f'Total Dataset Images:       {n_total:,}')
print(f'Training Split (70%):       {n_train:,} images ({len(train_loader)} batches)')
print(f'Validation Split (15%):     {n_val:,} images ({len(val_loader)} batches)')
print(f'Test Split (15%):           {n_test:,} images ({len(test_loader)} batches)')

### 3. Visualizing Sample Augmented Training Images
We inspect sample images passed through the training pipeline with random flips, rotations, and color jitter.

In [ ]:
images, labels, _ = next(iter(train_loader))
mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for i, ax in enumerate(axes.flat):
    img = images[i].numpy().transpose((1, 2, 0))
    img = np.clip(std * img + mean, 0, 1)
    ax.imshow(img)
    breed_title = class_names.get(labels[i].item(), f'Class {labels[i].item()}')
    ax.set_title(breed_title, fontsize=10, fontweight='bold')
    ax.axis('off')
plt.suptitle('Sample Training Images (Resized 224x224 & Augmented)', fontsize=13, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

### 4. EfficientNetV2 Architecture & Transfer Learning (Rubric Section 6)
We instantiate `EfficientNetV2-S` with pretrained ImageNet weights and adapt the classification head for 120 breeds.

In [ ]:
model = get_model(model_name='efficientnet_v2_s', num_classes=len(class_names), pretrained=True)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'Model Name:           {model.name}')
print(f'Total Parameters:     {total_params:,}')
print(f'Trainable Parameters: {trainable_params:,}')
print(f'Classifier Head:\n{model.classifier}')

### 5. Training History & Learning Curves (Rubric Section 7.3 & 8)
Plot the training vs. validation loss and accuracy curves across all training epochs.

In [ ]:
history_path = Path('history.json') if Path('history.json').exists() else Path('JP_Model/history.json')
if history_path.exists():
    with open(history_path, 'r', encoding='utf-8') as f:
        hist = json.load(f)
    epochs = range(1, len(hist['train_loss']) + 1)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
    ax1.plot(epochs, hist['train_loss'], 'o-', color='#2563eb', label='Train Loss')
    ax1.plot(epochs, hist['val_loss'], 's--', color='#dc2626', label='Val Loss')
    ax1.set_title('Cross Entropy Loss', fontweight='bold')
    ax1.set_xlabel('Epoch')
    ax1.legend()
    ax1.grid(True, linestyle=':', alpha=0.6)

    ax2.plot(epochs, hist['train_acc'], 'o-', color='#059669', label='Train Accuracy')
    ax2.plot(epochs, hist['val_acc'], 's--', color='#d97706', label='Val Accuracy')
    ax2.set_title('Top-1 Accuracy (%)', fontweight='bold')
    ax2.set_xlabel('Epoch')
    ax2.legend()
    ax2.grid(True, linestyle=':', alpha=0.6)
    plt.tight_layout()
    plt.show()
else:
    print('History file not found. Run python train.py to train the model.')

### 6. Held-Out Test Set Evaluation (Rubric Section 8)
Evaluate the best model checkpoint on the held-out test split, computing Top-1 and Top-5 Accuracy, Macro F1-Score, and Precision/Recall.

In [ ]:
checkpoint_p = Path('best_model.pth') if Path('best_model.pth').exists() else Path('JP_Model/best_model.pth')
if checkpoint_p.exists():
    metrics = evaluate_checkpoint(checkpoint_p, output_dir=checkpoint_p.parent)
else:
    print('Checkpoint not found. Run train.py to generate best_model.pth.')

### 7. Confusion Matrix & Error Analysis
We visualize the 120-breed confusion matrix and inspect correct predictions and error modes for qualitative analysis.

In [ ]:
cm_img_p = Path('confusion_matrix.png') if Path('confusion_matrix.png').exists() else Path('JP_Model/confusion_matrix.png')
pred_img_p = Path('sample_predictions.png') if Path('sample_predictions.png').exists() else Path('JP_Model/sample_predictions.png')

if cm_img_p.exists():
    plt.figure(figsize=(9, 7))
    plt.imshow(Image.open(cm_img_p))
    plt.axis('off')
    plt.title('120-Breed Confusion Matrix', fontweight='bold')
    plt.show()

if pred_img_p.exists():
    plt.figure(figsize=(12, 9))
    plt.imshow(Image.open(pred_img_p))
    plt.axis('off')
    plt.title('Sample Test Predictions & Error Analysis', fontweight='bold')
    plt.show()

### 8. Group Comparison & Conclusions
In the final report, this EfficientNetV2 model will be compared against:
- **Lindani's Baseline CNN**: Demonstrating the accuracy gains of pretrained weights and compound scaling over training from scratch.
- **Sulaiman's Swin Transformer**: Comparing hierarchical self-attention against convolutional inductive biases on fine-grained dog breed classification.